# Notebook Docente 03 — Clasificación no supervisada (K-means)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/miguepoloc/teledeteccion/blob/main/sesiones/sesion-02/docente/colab/03_clasificacion_no_supervisada.ipynb)

## Teledetección — Maestría en Ingeniería, Universidad del Magdalena
**Sesión 2 · Sábado 18 de julio de 2026 · Bloque 5**

---

**Este notebook es para que TÚ (el docente) lo proyectes en pantalla.** Versión
en Python/Colab del script `docente/gee/03_clasificacion_no_supervisada.js`.

### Importante — qué es esto y qué NO es
Esta es una clasificación **no supervisada** (K-means), generada automáticamente
SIN datos de entrenamiento reales (ROIs). Sirve como "resultado ya hecho" para
mostrar cómo se ve un mapa clasificado y discutir el concepto general.

**NO es** la clasificación supervisada validada del Artículo 1 (esa usa Random
Forest con ROIs reales en campo, OA≥85%, Kappa≥0.80) — esa la construirán los
estudiantes en la Sesión 3 con sus propios ROIs. Sé explícito con el grupo
sobre esta diferencia al mostrar la capa.

In [1]:
!pip install geemap --quiet
print("✓ Instalación completada")

✓ Instalación completada


In [2]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='teledeteccion-miguepoloc')

print("✓ Google Earth Engine inicializado correctamente")

✓ Google Earth Engine inicializado correctamente


## Paso 1 — Imagen base con índices como bandas de entrada

In [3]:
zona_estudio = ee.Geometry.Rectangle([-74.2, 10.5, -73.8, 11.0])

def enmascarar_nubes(imagen):
    scl = imagen.select('SCL')
    mascara = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return imagen.updateMask(mascara)

imagen = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(zona_estudio)
    .filterDate('2024-01-01', '2024-03-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
    .map(enmascarar_nubes)
    .median()
    .clip(zona_estudio)
)

ndvi = imagen.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndwi = imagen.normalizedDifference(['B8', 'B11']).rename('NDWI')

# Bandas de entrada al clasificador: reflectancia + índices
bandas_entrada = imagen.select(['B2', 'B3', 'B4', 'B8', 'B11']).addBands([ndvi, ndwi])

print("✓ Imagen e índices listos")

✓ Imagen e índices listos


## Paso 2 — Muestrear pixels y entrenar K-means (5 clusters)

In [4]:
# El clusterer necesita una muestra de pixels para "aprender" los grupos —
# esto NO son ROIs etiquetados, son solo puntos aleatorios sin clase
muestra = bandas_entrada.sample(region=zona_estudio, scale=20, numPixels=5000, seed=42)

numero_clusters = 5
clusterer = ee.Clusterer.wekaKMeans(numero_clusters).train(muestra)

clasificacion = bandas_entrada.cluster(clusterer)

print("✓ Clasificación no supervisada calculada")

✓ Clasificación no supervisada calculada


## Paso 3 — Mapa comparativo

In [5]:
paleta_clusters = ['#8B4513', '#DAA520', '#006400', '#90EE90', '#0000FF']

m = geemap.Map()
m.centerObject(zona_estudio, 12)
m.addLayer(imagen, {'bands': ['B8', 'B4', 'B3'], 'min': 0, 'max': 4000}, '1 - Falso Color (referencia)')
m.addLayer(clasificacion, {'min': 0, 'max': numero_clusters - 1, 'palette': paleta_clusters},
           '2 - Clasificación no supervisada (5 clusters)', False)
m.addLayerControl()
m

Map(center=[10.749994930545569, -73.99999999999935], controls=(WidgetControl(options=['position', 'transparent…

## Paso 4 — Área aproximada por cluster

In [6]:
area_por_cluster = (
    ee.Image.pixelArea().addBands(clasificacion)
    .reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName='cluster'),
        geometry=zona_estudio, scale=20, maxPixels=1e9
    )
    .getInfo()
)

print("=== Área por cluster (m²) ===")
for grupo in area_por_cluster['groups']:
    print(f"  Cluster {grupo['cluster']}: {grupo['sum']/1e6:.2f} km²")

=== Área por cluster (m²) ===
  Cluster 0: 546.71 km²
  Cluster 1: 198.79 km²
  Cluster 2: 268.12 km²
  Cluster 3: 615.50 km²
  Cluster 4: 790.63 km²


## Guion para la clase

Estos 5 clusters son grupos **estadísticos** de pixels espectralmente parecidos
— el algoritmo no sabe qué es "cacao" o "café", solo agrupa números similares.
Para saber qué cobertura real representa cada cluster, necesitarías visitar el
campo (verdad de terreno) o compararlo con un mapa ya validado.

**En la Sesión 3:** van a hacer el proceso completo — dibujar ROIs reales
etiquetados (cacao, café, bosque, pasturas, agua), entrenar un clasificador
**supervisado** (Random Forest) y calcular una matriz de confusión con
accuracy real. Hoy solo ven cómo se ve un mapa clasificado, como referencia
visual del Artículo 1.